# Champions Predictor - Parte 2: Simulación Monte Carlo

## Objetivo
Simular 10,000 torneos completos de Champions League para calcular probabilidades de campeón.

## Proceso
1. Cargar power ratings
2. Simular cada partido (ida + vuelta)
3. Simular cada ronda (octavos → cuartos → semis → final)
4. Ejecutar 10,000 simulaciones
5. Calcular probabilidades

In [8]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from collections import Counter
import random

## 1. Cargar Power Ratings

In [9]:
# Cargar ratings
df_ratings = pd.read_csv('../data/processed/power_ratings_final.csv')

# Crear diccionario de ratings
ratings_dict = dict(zip(df_ratings['Equipo'], df_ratings['power_rating']))

print("✅ Ratings cargados")
print(f"\nTotal equipos: {len(ratings_dict)}")
print("\nTop 5:")
print(df_ratings.head()[['ranking', 'Equipo', 'power_rating']])

✅ Ratings cargados

Total equipos: 16

Top 5:
   ranking           Equipo  power_rating
0        1    Bayern Munich   1963.480130
1        2          Arsenal   1895.024032
2        3        Barcelona   1870.061731
3        4      Real Madrid   1837.363398
4        5  Manchester City   1816.810440


## 2. Bracket Fijo de Champions

### Estructura del Bracket (FIJA - respeta sorteo real):

```
LADO A (Top Half):
├─ PSG vs Chelsea
├─ Liverpool vs Galatasaray
├─ Bayern vs Atalanta  
└─ Real Madrid vs Man City

LADO B (Bottom Half):
├─ Sporting vs Bodø/Glimt
├─ Arsenal vs Leverkusen
├─ Barcelona vs Newcastle
└─ Tottenham vs Atlético
```

**Los ganadores de cada lado solo se pueden encontrar en la FINAL**

In [10]:
# BRACKET FIJO - Respeta sorteo real de Champions

# LADO A - Top Half del bracket
octavos_lado_a = [
    ('Paris Saint-Germain', 'Chelsea'),           # Tie A1
    ('Liverpool', 'Galatasaray'),                 # Tie A2
    ('Bayern Munich', 'Atalanta'),                # Tie A3
    ('Real Madrid', 'Manchester City')            # Tie A4
]

# LADO B - Bottom Half del bracket
octavos_lado_b = [
    ('Sporting Lisboa', 'Bodø/Glimt'),            # Tie B1
    ('Arsenal', 'Bayer Leverkusen'),              # Tie B2
    ('Barcelona', 'Newcastle United'),            # Tie B3
    ('Tottenham Hotspur', 'Atlético Madrid')      # Tie B4
]

# Bracket completo
octavos_completo = octavos_lado_a + octavos_lado_b

print("⚔️ BRACKET DE OCTAVOS - CHAMPIONS LEAGUE 2024-25\n")
print("═" * 50)
print("\n🔴 LADO A (Top Half):")
for i, (eq1, eq2) in enumerate(octavos_lado_a, 1):
    print(f"  A{i}. {eq1} vs {eq2}")

print("\n🔵 LADO B (Bottom Half):")
for i, (eq1, eq2) in enumerate(octavos_lado_b, 1):
    print(f"  B{i}. {eq1} vs {eq2}")

print("\n" + "═" * 50)
print("\n⚠️ Los equipos del Lado A solo pueden enfrentar a equipos del Lado B en la FINAL")

⚔️ BRACKET DE OCTAVOS - CHAMPIONS LEAGUE 2024-25

══════════════════════════════════════════════════

🔴 LADO A (Top Half):
  A1. Paris Saint-Germain vs Chelsea
  A2. Liverpool vs Galatasaray
  A3. Bayern Munich vs Atalanta
  A4. Real Madrid vs Manchester City

🔵 LADO B (Bottom Half):
  B1. Sporting Lisboa vs Bodø/Glimt
  B2. Arsenal vs Bayer Leverkusen
  B3. Barcelona vs Newcastle United
  B4. Tottenham Hotspur vs Atlético Madrid

══════════════════════════════════════════════════

⚠️ Los equipos del Lado A solo pueden enfrentar a equipos del Lado B en la FINAL


## 3. Funciones de Simulación

### 3.1 Probabilidad de Victoria

In [11]:
def calcular_probabilidad(rating_a, rating_b):
    """Probabilidad de que A gane a B"""
    return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))

### 3.2 Simular un Partido Individual

In [12]:
def simular_partido(equipo_a, equipo_b, ratings_dict, es_local_a=True):
    """
    Simula un partido entre dos equipos
    
    Returns:
    --------
    tuple: (goles_a, goles_b)
    """
    rating_a = ratings_dict[equipo_a]
    rating_b = ratings_dict[equipo_b]
    
    # Ventaja de local (+50 puntos de rating)
    if es_local_a:
        rating_a += 50
    else:
        rating_b += 50
    
    # Probabilidad de victoria
    prob_a = calcular_probabilidad(rating_a, rating_b)
    
    # Simular resultado basado en probabilidad
    # Usamos distribución de Poisson para goles
    
    # Lambda base (goles esperados)
    lambda_a = 1.5 + (prob_a - 0.5) * 2  # Entre 0.5 y 2.5 goles
    lambda_b = 1.5 + (1 - prob_a - 0.5) * 2
    
    # Asegurar que lambdas sean positivos
    lambda_a = max(0.3, lambda_a)
    lambda_b = max(0.3, lambda_b)
    
    # Generar goles
    goles_a = np.random.poisson(lambda_a)
    goles_b = np.random.poisson(lambda_b)
    
    return goles_a, goles_b

### 3.3 Simular Eliminatoria (Ida + Vuelta)

In [13]:
def simular_eliminatoria(equipo_1, equipo_2, ratings_dict):
    """
    Simula eliminatoria completa (ida y vuelta)
    
    REGLAS UEFA 2021+: 
    - NO hay regla de gol de visitante
    - Si empate global → Prórroga (30 min)
    - Si sigue empate → Penales
    
    Returns:
    --------
    str: Nombre del equipo ganador
    """
    # IDA (equipo_1 local)
    goles_ida_1, goles_ida_2 = simular_partido(equipo_1, equipo_2, ratings_dict, es_local_a=True)
    
    # VUELTA (equipo_2 local)
    goles_vuelta_2, goles_vuelta_1 = simular_partido(equipo_2, equipo_1, ratings_dict, es_local_a=True)
    
    # Goles totales
    total_1 = goles_ida_1 + goles_vuelta_1
    total_2 = goles_ida_2 + goles_vuelta_2
    
    # Determinar ganador
    if total_1 > total_2:
        return equipo_1
    elif total_2 > total_1:
        return equipo_2
    else:
        # EMPATE EN GLOBAL → PRÓRROGA + PENALES
        # Simulamos prórroga (30 minutos extra)
        # Reducimos intensidad de ataque (equipos más cansados)
        goles_prorroga_1 = simular_goles_prorroga(equipo_1, equipo_2, ratings_dict)
        goles_prorroga_2 = simular_goles_prorroga(equipo_2, equipo_1, ratings_dict)
        
        if goles_prorroga_1 > goles_prorroga_2:
            return equipo_1
        elif goles_prorroga_2 > goles_prorroga_1:
            return equipo_2
        else:
            # PENALES (con sesgo por rating)
            prob_1 = calcular_probabilidad(ratings_dict[equipo_1], ratings_dict[equipo_2])
            return equipo_1 if random.random() < prob_1 else equipo_2
 
 
def simular_goles_prorroga(equipo_ataque, equipo_defensa, ratings_dict):
    """
    Simula goles en prórroga (30 min = 1/3 de partido)
    Reduce lambda por cansancio
    """
    rating_ataque = ratings_dict[equipo_ataque]
    rating_defensa = ratings_dict[equipo_defensa]
    
    # Fuerza ofensiva (reducida por cansancio)
    lambda_goles = ((rating_ataque - 1600) / 400) * 1.5 * 0.33 * 0.7  # Factor 0.7 = cansancio
    lambda_goles = max(0.1, lambda_goles)  # Mínimo 0.1
    
    return np.random.poisson(lambda_goles)

### 3.4 Simular Torneo Completo 

In [14]:
def simular_torneo_completo(octavos_lado_a, octavos_lado_b, ratings_dict):
    """
    Simula todo el torneo respetando bracket fijo
    
    Bracket:
    - Lado A: 4 ties → 2 cuartos → 1 semi → 1 finalista
    - Lado B: 4 ties → 2 cuartos → 1 semi → 1 finalista
    - Final: Finalista A vs Finalista B
    
    Returns:
    --------
    dict: Resultados por ronda + campeón
    """
    resultados = {
        
        'octavos_a': [],
        'octavos_b': [],
        'cuartos_a': [],
        'cuartos_b': [],
        'semi_a': None,
        'semi_b': None,
        'final': None,
        'campeon': None
    }
    
    # ============================================
    # OCTAVOS DE FINAL - LADO A
    # ============================================
    clasificados_cuartos_a = []
    for eq1, eq2 in octavos_lado_a:
        ganador = simular_eliminatoria(eq1, eq2, ratings_dict)
        clasificados_cuartos_a.append(ganador)
        resultados['octavos_a'].append((eq1, eq2, ganador))
    
    # ============================================
    # OCTAVOS DE FINAL - LADO B
    # ============================================
    clasificados_cuartos_b = []
    for eq1, eq2 in octavos_lado_b:
        ganador = simular_eliminatoria(eq1, eq2, ratings_dict)
        clasificados_cuartos_b.append(ganador)
        resultados['octavos_b'].append((eq1, eq2, ganador))
    
    # ============================================
    # CUARTOS DE FINAL - LADO A
    # ============================================
    # A1 ganador vs A2 ganador
    # A3 ganador vs A4 ganador
    clasificados_semi_a = []
    for i in range(0, 4, 2):
        eq1 = clasificados_cuartos_a[i]
        eq2 = clasificados_cuartos_a[i + 1]
        ganador = simular_eliminatoria(eq1, eq2, ratings_dict)
        clasificados_semi_a.append(ganador)
        resultados['cuartos_a'].append((eq1, eq2, ganador))
    
    # ============================================
    # CUARTOS DE FINAL - LADO B
    # ============================================
    # B1 ganador vs B2 ganador
    # B3 ganador vs B4 ganador
    clasificados_semi_b = []
    for i in range(0, 4, 2):
        eq1 = clasificados_cuartos_b[i]
        eq2 = clasificados_cuartos_b[i + 1]
        ganador = simular_eliminatoria(eq1, eq2, ratings_dict)
        clasificados_semi_b.append(ganador)
        resultados['cuartos_b'].append((eq1, eq2, ganador))
    
    # ============================================
    # SEMIFINAL - LADO A
    # ============================================
    eq1_a = clasificados_semi_a[0]
    eq2_a = clasificados_semi_a[1]
    finalista_a = simular_eliminatoria(eq1_a, eq2_a, ratings_dict)
    resultados['semi_a'] = (eq1_a, eq2_a, finalista_a)
    
    # ============================================
    # SEMIFINAL - LADO B
    # ============================================
    eq1_b = clasificados_semi_b[0]
    eq2_b = clasificados_semi_b[1]
    finalista_b = simular_eliminatoria(eq1_b, eq2_b, ratings_dict)
    resultados['semi_b'] = (eq1_b, eq2_b, finalista_b)
    
    # ============================================
    # FINAL - Finalista A vs Finalista B
    # ============================================
    # Partido único, sin ventaja local
    prob_a = calcular_probabilidad(ratings_dict[finalista_a], ratings_dict[finalista_b])
    campeon = finalista_a if random.random() < prob_a else finalista_b
    
    resultados['final'] = (finalista_a, finalista_b, campeon)
    resultados['campeon'] = campeon
    
    return resultados

## 4. Test - Simular UN Torneo

Antes de hacer 10,000, probamos con 1:

In [15]:
# Simular un torneo de prueba
print("🎲 SIMULACIÓN DE PRUEBA - UN TORNEO:\n")
print("═" * 60)

resultado_test = simular_torneo_completo(octavos_lado_a, octavos_lado_b, ratings_dict)

print("\n🔴 OCTAVOS - LADO A:")
for eq1, eq2, ganador in resultado_test['octavos_a']:
    print(f"  {eq1} vs {eq2} → Pasa: {ganador}")

print("\n🔵 OCTAVOS - LADO B:")
for eq1, eq2, ganador in resultado_test['octavos_b']:
    print(f"  {eq1} vs {eq2} → Pasa: {ganador}")

print("\n" + "═" * 60)

print("\n🔴 CUARTOS - LADO A:")
for eq1, eq2, ganador in resultado_test['cuartos_a']:
    print(f"  {eq1} vs {eq2} → Pasa: {ganador}")

print("\n🔵 CUARTOS - LADO B:")
for eq1, eq2, ganador in resultado_test['cuartos_b']:
    print(f"  {eq1} vs {eq2} → Pasa: {ganador}")

print("\n" + "═" * 60)

print("\n🔴 SEMIFINAL - LADO A:")
eq1, eq2, ganador = resultado_test['semi_a']
print(f"  {eq1} vs {eq2} → Finalista: {ganador}")

print("\n🔵 SEMIFINAL - LADO B:")
eq1, eq2, ganador = resultado_test['semi_b']
print(f"  {eq1} vs {eq2} → Finalista: {ganador}")

print("\n" + "═" * 60)

print("\n🏆 FINAL:")
eq1, eq2, campeon = resultado_test['final']
print(f"  {eq1} (Lado A) vs {eq2} (Lado B)")
print(f"\n  ⭐ CAMPEÓN: {campeon}")

print("\n" + "═" * 60)

🎲 SIMULACIÓN DE PRUEBA - UN TORNEO:

════════════════════════════════════════════════════════════

🔴 OCTAVOS - LADO A:
  Paris Saint-Germain vs Chelsea → Pasa: Paris Saint-Germain
  Liverpool vs Galatasaray → Pasa: Galatasaray
  Bayern Munich vs Atalanta → Pasa: Bayern Munich
  Real Madrid vs Manchester City → Pasa: Manchester City

🔵 OCTAVOS - LADO B:
  Sporting Lisboa vs Bodø/Glimt → Pasa: Bodø/Glimt
  Arsenal vs Bayer Leverkusen → Pasa: Arsenal
  Barcelona vs Newcastle United → Pasa: Newcastle United
  Tottenham Hotspur vs Atlético Madrid → Pasa: Atlético Madrid

════════════════════════════════════════════════════════════

🔴 CUARTOS - LADO A:
  Paris Saint-Germain vs Galatasaray → Pasa: Paris Saint-Germain
  Bayern Munich vs Manchester City → Pasa: Bayern Munich

🔵 CUARTOS - LADO B:
  Bodø/Glimt vs Arsenal → Pasa: Arsenal
  Newcastle United vs Atlético Madrid → Pasa: Atlético Madrid

════════════════════════════════════════════════════════════

🔴 SEMIFINAL - LADO A:
  Paris Saint-G

## 5. MONTE CARLO - 10,000 Simulaciones

### ⚠️ ESTO TARDA 2-3 MINUTOS

In [16]:
# Configuración
NUM_SIMULACIONES = 10000

print(f"🎲 Ejecutando {NUM_SIMULACIONES:,} simulaciones...")
print("⏱️ Esto puede tardar 2-3 minutos...\n")

# Contadores
campeones = []
finalistas = []
semifinalistas = []
cuartofinalistas = []

# Ejecutar simulaciones
for i in range(NUM_SIMULACIONES):
    if (i + 1) % 2000 == 0:
        print(f"  Progreso: {i+1:,}/{NUM_SIMULACIONES:,} ({(i+1)/NUM_SIMULACIONES*100:.1f}%)")
    
    resultado = simular_torneo_completo(octavos_lado_a, octavos_lado_b, ratings_dict)
    
    # Registrar campeón
    campeones.append(resultado['campeon'])
    
    # Registrar finalistas
    eq1, eq2, _ = resultado['final']
    finalistas.extend([eq1, eq2])
    
    # Registrar semifinalistas
    eq1_a, eq2_a, _ = resultado['semi_a']
    eq1_b, eq2_b, _ = resultado['semi_b']
    semifinalistas.extend([eq1_a, eq2_a, eq1_b, eq2_b])
    
    # Registrar cuartofinalistas
    for eq1, eq2, _ in resultado['cuartos_a']:
        cuartofinalistas.extend([eq1, eq2])
    for eq1, eq2, _ in resultado['cuartos_b']:
        cuartofinalistas.extend([eq1, eq2])

print(f"\n✅ {NUM_SIMULACIONES:,} simulaciones completadas!")

🎲 Ejecutando 10,000 simulaciones...
⏱️ Esto puede tardar 2-3 minutos...

  Progreso: 2,000/10,000 (20.0%)
  Progreso: 4,000/10,000 (40.0%)
  Progreso: 6,000/10,000 (60.0%)
  Progreso: 8,000/10,000 (80.0%)
  Progreso: 10,000/10,000 (100.0%)

✅ 10,000 simulaciones completadas!


## 6. Análisis de Resultados

In [17]:
# Contar frecuencias
contador_campeones = Counter(campeones)
contador_finalistas = Counter(finalistas)
contador_semis = Counter(semifinalistas)
contador_cuartos = Counter(cuartofinalistas)

# Crear DataFrame de probabilidades
equipos_unicos = list(ratings_dict.keys())

probabilidades = []
for equipo in equipos_unicos:
    prob_campeon = (contador_campeones.get(equipo, 0) / NUM_SIMULACIONES) * 100
    prob_final = (contador_finalistas.get(equipo, 0) / (NUM_SIMULACIONES * 2)) * 100
    prob_semi = (contador_semis.get(equipo, 0) / (NUM_SIMULACIONES * 4)) * 100
    prob_cuartos = (contador_cuartos.get(equipo, 0) / (NUM_SIMULACIONES * 8)) * 100
    
    probabilidades.append({
        'Equipo': equipo,
        'Prob_Campeon': prob_campeon,
        'Prob_Final': prob_final,
        'Prob_Semi': prob_semi,
        'Prob_Cuartos': prob_cuartos,
        'Rating': ratings_dict[equipo]
    })

df_probabilidades = pd.DataFrame(probabilidades)
df_probabilidades = df_probabilidades.sort_values('Prob_Campeon', ascending=False).reset_index(drop=True)
df_probabilidades['Ranking'] = range(1, len(df_probabilidades) + 1)

print("\n🏆 PROBABILIDADES DE CAMPEON - Champions League 2024-25\n")
print(df_probabilidades[['Ranking', 'Equipo', 'Prob_Campeon', 'Prob_Final', 'Prob_Semi']].to_string(index=False))


🏆 PROBABILIDADES DE CAMPEON - Champions League 2024-25

 Ranking              Equipo  Prob_Campeon  Prob_Final  Prob_Semi
       1       Bayern Munich         35.67      27.115    16.3625
       2             Arsenal         18.12      17.870    13.8925
       3           Barcelona         15.81      16.335    14.9325
       4     Sporting Lisboa          6.26       7.805     7.8850
       5         Real Madrid          5.01       5.440     4.4000
       6 Paris Saint-Germain          4.40       5.130     8.8000
       7     Manchester City          4.00       4.315     3.3750
       8     Atlético Madrid          3.19       4.395     6.5000
       9           Liverpool          2.03       2.795     6.0075
      10             Chelsea          1.95       2.650     5.7000
      11         Galatasaray          1.23       1.780     4.4925
      12    Bayer Leverkusen          0.75       1.380     2.0350
      13    Newcastle United          0.57       1.075     2.0400
      14           

## 7. Guardar Resultados

In [18]:
# Guardar probabilidades
df_probabilidades.to_csv('../data/processed/probabilidades_campeon.csv', index=False)
print("\n✅ Resultados guardados en: data/processed/probabilidades_campeon.csv")


✅ Resultados guardados en: data/processed/probabilidades_campeon.csv


## 8. Insights Clave

In [19]:
print("\n📊 INSIGHTS CLAVE:\n")

top_3 = df_probabilidades.head(3)

print("Top 3 Favoritos:")
for idx, row in top_3.iterrows():
    print(f"  {row['Ranking']}. {row['Equipo']}: {row['Prob_Campeon']:.1f}% de ser campeón")

print(f"\nLos 3 favoritos acumulan: {top_3['Prob_Campeon'].sum():.1f}% de probabilidad")

# Dark horses (equipos con más prob de lo esperado por rating)
df_probabilidades['diff_rating_prob'] = df_probabilidades['Prob_Campeon'] - (df_probabilidades['Rating'] - 1600) / 4
dark_horses = df_probabilidades.nlargest(3, 'diff_rating_prob')[['Equipo', 'Prob_Campeon', 'Rating']]

print("\nDark Horses (superan expectativas):")
for idx, row in dark_horses.iterrows():
    print(f"  {row['Equipo']}: {row['Prob_Campeon']:.1f}%")


📊 INSIGHTS CLAVE:

Top 3 Favoritos:
  1. Bayern Munich: 35.7% de ser campeón
  2. Arsenal: 18.1% de ser campeón
  3. Barcelona: 15.8% de ser campeón

Los 3 favoritos acumulan: 69.6% de probabilidad

Dark Horses (superan expectativas):
  Tottenham Hotspur: 0.2%
  Bodø/Glimt: 0.3%
  Newcastle United: 0.6%


## ✅ Simulación Completada

**Próximo paso:** Notebook 3 - Visualizaciones finales